<a href="https://colab.research.google.com/github/Drewbits/petrophysical-data-quality-workflow/blob/main/04_Standardize_Curves_and_Units.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Petrophysical Data Quality Workflow

## Notebook 4 – Curve and Unit Standardization

**Author:** Andrew Hind

### Business Objective

Petrophysical datasets frequently contain inconsistent curve names, unit notation, scaling conventions, and incomplete metadata across wells and vendors. These inconsistencies must be standardized before measurements can be reliably compared, quality controlled, analyzed, or loaded into a relational database.

### Technical Objectives

- Define canonical units for commonly used petrophysical measurements
- Separate notation changes from numerical unit conversions
- Preserve original source metadata
- Apply controlled standardization rules
- Document all transformations
- Validate standardized results
- Identify unresolved metadata requiring manual review

### Standardization Principles

1. Preserve the original source metadata.
2. Never change numerical values for notation-only differences.
3. Apply numerical conversions only through explicit conversion rules.
4. Record the transformation applied to each measurement.
5. Flag ambiguous metadata rather than silently correcting it.

### Deliverables

- Canonical unit dictionary
- Unit conversion rules
- Standardized metadata catalog
- Transformation audit fields
- Unresolved exception table

In [63]:
!pip install -q dlisio

In [64]:
from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd

from dlisio import dlis

drive.mount("/content/drive")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment ready.


In [65]:
print(dlis)

<module 'dlisio.dlis' from '/usr/local/lib/python3.12/dist-packages/dlisio/dlis/__init__.py'>


In [66]:
PROJECT_OUTPUTS = Path(
    "/content/drive/MyDrive/Datasets/petrophysical-data-quality-workflow/outputs"
)

PROJECT_OUTPUTS.mkdir(
    parents=True,
    exist_ok=True
)

print(PROJECT_OUTPUTS)
print(f"Directory exists: {PROJECT_OUTPUTS.exists()}")

/content/drive/MyDrive/Datasets/petrophysical-data-quality-workflow/outputs
Directory exists: True


Cell 3 — Define the source dataset

Use the actual dataset root we established earlier:

In [67]:
DATASET_ROOT = Path(
    "/content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE"
)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")

Dataset root: /content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE
Dataset exists: True


In [68]:
dlis_files = sorted(
    path
    for path in DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.upper() == ".DLIS"
)

print(f"DLIS files found: {len(dlis_files)}")

for path in dlis_files[:10]:
    print(path.relative_to(DATASET_ROOT))

DLIS files found: 34
15_9-F-1/WLC_COMPOSITE_1.DLIS
15_9-F-1/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 A/WLC_COMPOSITE_1.DLIS
15_9-F-1 A/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 B/WLC_COMPOSITE_1.DLIS
15_9-F-1 B/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-1 C/WLC_COMPOSITE_1.DLIS
15_9-F-1 C/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS
15_9-F-10/WLC_COMPOSITE_1.DLIS
15_9-F-10/WLC_PETROPHYSICAL_COMPOSITE_1.DLIS


Cell 5 — Rebuild the metadata catalog

Because we're deliberately making Notebook 4 independent of Notebook 3's CSV export, we'll reconstruct the metadata directly from the DLIS source.

## 1. Rebuild Source Metadata Catalog

The DLIS metadata catalog is reconstructed directly from the source archive so that this notebook remains reproducible without depending on intermediate files created by previous notebooks.

In [69]:
catalog_records = []
load_errors = []

for file_path in dlis_files:

    well_name = file_path.parent.name

    try:
        physical_file = dlis.load(file_path)

        for logical_index, logical_file in enumerate(physical_file):

            for frame_index, frame in enumerate(logical_file.frames):

                for channel in frame.channels:

                    catalog_records.append(
                        {
                            "well_name": well_name,
                            "file_name": file_path.name,
                            "relative_path": str(
                                file_path.relative_to(DATASET_ROOT)
                            ),
                            "logical_file_index": logical_index,
                            "frame_index": frame_index,
                            "frame_name": frame.name,
                            "index_type": frame.index_type,
                            "mnemonic": channel.name,
                            "long_name": channel.long_name,
                            "source_unit": channel.units,
                        }
                    )

    except Exception as error:

        load_errors.append(
            {
                "well_name": well_name,
                "file_name": file_path.name,
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

We're explicitly calling it source_unit because we're about to introduce a separate standardized unit. That makes provenance clearer.

In [70]:
catalog_df = pd.DataFrame(catalog_records)
errors_df = pd.DataFrame(load_errors)

print(f"Catalog rows created: {len(catalog_df):,}")
print(f"Files with loading errors: {len(errors_df)}")
print(f"Unique wells: {catalog_df['well_name'].nunique()}")
print(f"Unique mnemonics: {catalog_df['mnemonic'].nunique()}")

Catalog rows created: 1,078
Files with loading errors: 0
Unique wells: 17
Unique mnemonics: 229


## 2. Define Canonical Unit Standards

Source DLIS files may represent the same physical measurement using different unit spellings, capitalization, or notation. This section defines canonical units while preserving the original source unit for traceability.

In [71]:
unit_mapping = {
    # Gamma ray
    "GAPI": "gAPI",
    "gAPI": "gAPI",

    # Resistivity
    "OHM.M": "ohm.m",
    "OHMM": "ohm.m",
    "ohm.m": "ohm.m",

    # Sonic slowness
    "US/F": "us/ft",
    "US/FT": "us/ft",
    "us/ft": "us/ft",

    # Length / diameter
    "IN": "in",
    "in": "in",

    # Density
    "G/CC": "g/cm3",
    "g/cm3": "g/cm3",

    # Fractional porosity
    "V/V": "v/v",
    "v/v": "v/v",

    # Photoelectric factor
    "B/E": "b/e",
    "b/e": "b/e",

    # Rate of penetration
    "M/H": "m/h",
    "m/h": "m/h",

    # Temperature
    "DEGC": "degC",
    "degC": "degC",

    # Time
    "S": "s",
    "s": "s",

    # Dimensionless
    "unitless": "unitless",
    "": "unknown",
}

In [72]:
catalog_df["canonical_unit"] = (
    catalog_df["source_unit"]
    .fillna("")
    .map(unit_mapping)
    .fillna(catalog_df["source_unit"])
)

In [73]:
catalog_df[
    [
        "mnemonic",
        "source_unit",
        "canonical_unit"
    ]
].drop_duplicates().head(50)

,mnemonic,source_unit,canonical_unit
0,DEPTH,mm,mm
1,GR,gAPI,gAPI
2,CALI,in,in
3,RDEP,ohm.m,ohm.m
4,RMED,ohm.m,ohm.m
5,DEN,g/cm3,g/cm3
6,DENC,g/cm3,g/cm3
7,PEF,b/e,b/e
8,NEU,v/v,v/v
9,AC,us/ft,us/ft


In [74]:
remaining_unit_variation = (
    catalog_df.groupby("mnemonic")["canonical_unit"]
    .nunique()
    .sort_values(ascending=False)
)

remaining_unit_variation[
    remaining_unit_variation > 1
]

,canonical_unit
mnemonic,
DXFE_WAL,2
DWTI_WAL,2
DWSU_WAL,2
CRPM,2
DEPTH,2
DWAL_WAL,2
DWCA_WAL,2
DWSI_WAL,2
DWFE_WAL,2


## 3. Define Numerical Unit Conversion Rules

Notation standardization changes metadata labels without modifying measurement values.

True unit conversions are handled separately because they require modification of the numerical data. Each conversion rule defines the source unit, target unit, and conversion factor required to produce the canonical measurement.

The original source unit is retained for traceability.

In [75]:
conversion_candidates = (
    catalog_df[
        catalog_df["mnemonic"].isin(["DEPTH", "TNPH"])
    ][
        [
            "well_name",
            "file_name",
            "mnemonic",
            "long_name",
            "source_unit",
            "canonical_unit",
        ]
    ]
    .drop_duplicates()
    .sort_values(["mnemonic", "source_unit", "well_name"])
)

conversion_candidates

,well_name,file_name,mnemonic,long_name,source_unit,canonical_unit
616,15_9-F-15,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
628,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
689,15_9-F-15 A,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
701,15_9-F-15 A,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
747,15_9-F-15 B,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
809,15_9-F-15 C,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
881,15_9-F-15 D,WLC_COMPOSITE_2.DLIS,DEPTH,,0.1 in,0.1 in
971,15_9-F-5,WLC_COMPOSITE_1.DLIS,DEPTH,,0.1 in,0.1 in
983,15_9-F-5,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,,0.1 in,0.1 in
1031,15_9-F-9,WLC_COMPOSITE_1.DLIS,DEPTH,,0.1 in,0.1 in


In [76]:
conversion_rules = pd.DataFrame(
    [
        {
            "mnemonic": "DEPTH",
            "source_unit": "0.1 in",
            "target_unit": "mm",
            "conversion_factor": 2.54,
            "conversion_offset": 0.0,
            "rule_description": "Convert tenths of an inch to millimeters",
        },
        {
            "mnemonic": "TNPH",
            "source_unit": "PU",
            "target_unit": "v/v",
            "conversion_factor": 0.01,
            "conversion_offset": 0.0,
            "rule_description": "Convert porosity units to decimal volume fraction",
        },
    ]
)

conversion_rules

,mnemonic,source_unit,target_unit,conversion_factor,conversion_offset,rule_description
0,DEPTH,0.1 in,mm,2.54,0.0,Convert tenths of an inch to millimeters
1,TNPH,PU,v/v,0.01,0.0,Convert porosity units to decimal volume fraction


In [77]:
catalog_df = catalog_df.merge(
    conversion_rules,
    on=["mnemonic", "source_unit"],
    how="left",
)

In [78]:
catalog_df["conversion_required"] = (
    catalog_df["conversion_factor"].notna()
)

In [79]:
catalog_df[
    catalog_df["conversion_required"]
][
    [
        "well_name",
        "file_name",
        "mnemonic",
        "source_unit",
        "target_unit",
        "conversion_factor",
        "rule_description",
    ]
].drop_duplicates()

,well_name,file_name,mnemonic,source_unit,target_unit,conversion_factor,rule_description
616,15_9-F-15,WLC_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
628,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
689,15_9-F-15 A,WLC_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
701,15_9-F-15 A,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
747,15_9-F-15 B,WLC_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
800,15_9-F-15 B,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,TNPH,PU,v/v,0.01,Convert porosity units to decimal volume fraction
809,15_9-F-15 C,WLC_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
881,15_9-F-15 D,WLC_COMPOSITE_2.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
971,15_9-F-5,WLC_COMPOSITE_1.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters
983,15_9-F-5,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,DEPTH,0.1 in,mm,2.54,Convert tenths of an inch to millimeters


In [80]:
print(
    "Catalog records requiring numerical conversion:",
    catalog_df["conversion_required"].sum()
)

Catalog records requiring numerical conversion: 14


Step 5 — Verify the 14 records by rule

In [81]:
conversion_summary = (
    catalog_df[
        catalog_df["conversion_required"]
    ]
    .groupby(
        [
            "mnemonic",
            "source_unit",
            "target_unit",
            "conversion_factor",
        ]
    )
    .agg(
        records=("mnemonic", "size"),
        wells=("well_name", "nunique"),
    )
    .reset_index()
)

conversion_summary

,mnemonic,source_unit,target_unit,conversion_factor,records,wells
0,DEPTH,0.1 in,mm,2.54,13,8
1,TNPH,PU,v/v,0.01,1,1


## 4. Classify Metadata Standardization Actions

Not all metadata inconsistencies require numerical conversion. Some represent notation differences, missing metadata, malformed metadata, or ambiguous source information.

These conditions are classified separately so that automated corrections are limited to cases where the intended standardization is well defined.

In [82]:
def classify_standardization(row):

    # Numerical conversion rule exists
    if row["conversion_required"]:
        return "numerical_conversion"

    # Source unit is missing
    if pd.isna(row["source_unit"]) or row["source_unit"] == "":
        return "missing_unit_metadata"

    # Known malformed weight-fraction notation
    if row["source_unit"] == "KGF/":
        return "malformed_unit_metadata"

    # Unit notation changed but numerical values do not
    if row["source_unit"] != row["canonical_unit"]:
        return "notation_standardized"

    return "already_standard"

In [83]:
catalog_df["standardization_action"] = catalog_df.apply(
    classify_standardization,
    axis=1
)

In [84]:
catalog_df["standardization_action"].value_counts()

,count
standardization_action,
already_standard,691
notation_standardized,321
missing_unit_metadata,40
numerical_conversion,14
malformed_unit_metadata,12


In [85]:
notation_changes = (
    catalog_df[
        catalog_df["standardization_action"]
        == "notation_standardized"
    ]
    .groupby(
        ["source_unit", "canonical_unit"]
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

notation_changes

,source_unit,canonical_unit,records
7,OHMM,ohm.m,84
2,G/CC,g/cm3,75
4,IN,in,44
0,B/E,b/e,36
9,US/F,us/ft,24
3,GAPI,gAPI,18
5,M/H,m/h,14
11,V/V,v/v,13
8,S,s,6
1,DEGC,degC,4


In [86]:
print(f"Total catalog records: {len(catalog_df):,}")
print(
    "Total classified records:",
    catalog_df["standardization_action"].value_counts().sum()
)

Total catalog records: 1,078
Total classified records: 1078


In [87]:
notation_changes = (
    catalog_df[
        catalog_df["standardization_action"]
        == "notation_standardized"
    ]
    .groupby(
        ["source_unit", "canonical_unit"]
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

notation_changes

,source_unit,canonical_unit,records
7,OHMM,ohm.m,84
2,G/CC,g/cm3,75
4,IN,in,44
0,B/E,b/e,36
9,US/F,us/ft,24
3,GAPI,gAPI,18
5,M/H,m/h,14
11,V/V,v/v,13
8,S,s,6
1,DEGC,degC,4


## 5. Review Missing and Malformed Metadata

Some records cannot be safely standardized through notation cleanup or numerical conversion alone.

These records are separated into:

- missing unit metadata
- malformed unit metadata

They remain flagged for review rather than being silently corrected.

In [88]:
unresolved_metadata_df = (
    catalog_df[
        catalog_df["standardization_action"].isin(
            [
                "missing_unit_metadata",
                "malformed_unit_metadata",
            ]
        )
    ][
        [
            "well_name",
            "file_name",
            "mnemonic",
            "long_name",
            "source_unit",
            "canonical_unit",
            "standardization_action",
        ]
    ]
    .sort_values(
        [
            "standardization_action",
            "mnemonic",
            "well_name",
        ]
    )
    .reset_index(drop=True)
)

print(f"Unresolved metadata records: {len(unresolved_metadata_df):,}")

unresolved_metadata_df.head(30)

Unresolved metadata records: 52


,well_name,file_name,mnemonic,long_name,source_unit,canonical_unit,standardization_action
0,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWAL_WAL,Dry Weight Fraction Pseudo Aluminum (SpectroLi...,KGF/,KGF/,malformed_unit_metadata
1,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWCA_WAL,Dry Weight Fraction Calcium (SpectroLith WALK2...,KGF/,KGF/,malformed_unit_metadata
2,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWFE_WAL,Dry Weight Fraction Iron + 0.14 Aluminum (Spec...,KGF/,KGF/,malformed_unit_metadata
3,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWSI_WAL,Dry Weight Fraction Silicon (SpectroLith WALK2...,KGF/,KGF/,malformed_unit_metadata
4,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWSU_WAL,Dry Weight Fraction Sulfur (SpectroLith WALK2 ...,KGF/,KGF/,malformed_unit_metadata
5,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DWTI_WAL,Dry Weight Fraction Titanium (SpectroLith WALK...,KGF/,KGF/,malformed_unit_metadata
6,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,DXFE_WAL,Dry Weight Fraction Excess Iron (SpectroLith W...,KGF/,KGF/,malformed_unit_metadata
7,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,WCAR,Weight Percent Of Carbonate,KGF/,KGF/,malformed_unit_metadata
8,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,WCLA,Weight Percent Of Clay,KGF/,KGF/,malformed_unit_metadata
9,15_9-F-15,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,WPYR,Weight Percent Of Pyrite,KGF/,KGF/,malformed_unit_metadata


## 6. Build Standardized Metadata and Audit Fields

Final standardized metadata are assigned only where the transformation is unambiguous.

Records with missing or malformed source metadata remain unresolved and are flagged for manual review. Original source units are preserved throughout the workflow to maintain provenance.

In [89]:
def assign_standardized_unit(row):

    # Numerical conversion produces a new target unit
    if row["standardization_action"] == "numerical_conversion":
        return row["target_unit"]

    # Safe metadata-only standardization
    if row["standardization_action"] in [
        "already_standard",
        "notation_standardized",
    ]:
        return row["canonical_unit"]

    # Do not invent units for unresolved metadata
    return pd.NA


catalog_df["standardized_unit"] = catalog_df.apply(
    assign_standardized_unit,
    axis=1
)

Step 2 — Add manual-review flag

In [90]:
catalog_df["manual_review_required"] = (
    catalog_df["standardization_action"].isin(
        [
            "missing_unit_metadata",
            "malformed_unit_metadata",
        ]
    )
)

In [91]:
catalog_df["manual_review_required"].value_counts()

,count
manual_review_required,
False,1026
True,52


Step 3 — Add transformation audit descriptions

In [92]:
def create_audit_note(row):

    action = row["standardization_action"]

    if action == "already_standard":
        return "No transformation required"

    if action == "notation_standardized":
        return (
            f"Unit notation standardized: "
            f"{row['source_unit']} -> {row['canonical_unit']}"
        )

    if action == "numerical_conversion":
        return (
            f"Numerical conversion required: "
            f"{row['source_unit']} -> {row['target_unit']}; "
            f"value = source × {row['conversion_factor']} "
            f"+ {row['conversion_offset']}"
        )

    if action == "missing_unit_metadata":
        return "Source unit missing; manual review required"

    if action == "malformed_unit_metadata":
        return (
            f"Malformed source unit '{row['source_unit']}'; "
            "manual review required"
        )

    return "Unclassified"


catalog_df["audit_note"] = catalog_df.apply(
    create_audit_note,
    axis=1
)

In [93]:
audit_sample = (
    catalog_df.groupby(
        "standardization_action",
        group_keys=False
    )
    .head(2)
    [
        [
            "well_name",
            "mnemonic",
            "source_unit",
            "canonical_unit",
            "standardized_unit",
            "standardization_action",
            "manual_review_required",
            "audit_note",
        ]
    ]
)

audit_sample

,well_name,mnemonic,source_unit,canonical_unit,standardized_unit,standardization_action,manual_review_required,audit_note
0,15_9-F-1,DEPTH,mm,mm,mm,already_standard,False,No transformation required
1,15_9-F-1,GR,gAPI,gAPI,gAPI,already_standard,False,No transformation required
270,15_9-F-1 C,AZRIT1T2,,unknown,<NA>,missing_unit_metadata,True,Source unit missing; manual review required
271,15_9-F-1 C,AZRTBM,,unknown,<NA>,missing_unit_metadata,True,Source unit missing; manual review required
304,15_9-F-10,GR,GAPI,gAPI,gAPI,notation_standardized,False,Unit notation standardized: GAPI -> gAPI
305,15_9-F-10,CALI,IN,in,in,notation_standardized,False,Unit notation standardized: IN -> in
616,15_9-F-15,DEPTH,0.1 in,0.1 in,mm,numerical_conversion,False,Numerical conversion required: 0.1 in -> mm; v...
628,15_9-F-15,DEPTH,0.1 in,0.1 in,mm,numerical_conversion,False,Numerical conversion required: 0.1 in -> mm; v...
645,15_9-F-15,DWAL_WAL,KGF/,KGF/,<NA>,malformed_unit_metadata,True,Malformed source unit 'KGF/'; manual review re...
646,15_9-F-15,DWCA_WAL,KGF/,KGF/,<NA>,malformed_unit_metadata,True,Malformed source unit 'KGF/'; manual review re...


This is one of the strongest tables in the project because it demonstrates provenance.

Step 4 — Create the final Milestone 4 summary

In [94]:
standardization_summary = (
    catalog_df.groupby("standardization_action")
    .agg(
        records=("mnemonic", "count"),
        unique_mnemonics=("mnemonic", "nunique"),
        affected_wells=("well_name", "nunique"),
    )
    .sort_values("records", ascending=False)
)

standardization_summary

,records,unique_mnemonics,affected_wells
standardization_action,,,
already_standard,691,182,16
notation_standardized,321,63,8
missing_unit_metadata,40,27,7
numerical_conversion,14,2,8
malformed_unit_metadata,12,12,1


In [95]:
print(f"Total metadata records: {len(catalog_df):,}")
print(
    "Automatically resolved:",
    (~catalog_df["manual_review_required"]).sum()
)
print(
    "Manual review required:",
    catalog_df["manual_review_required"].sum()
)
print(
    "Numerical conversions defined:",
    catalog_df["conversion_required"].sum()
)

Total metadata records: 1,078
Automatically resolved: 1026
Manual review required: 52
Numerical conversions defined: 14


Step 7 — Validate the conversion mathematics We don't need to process all DLIS measurement data yet. We'll test representative values first.

## 7. Validate Numerical Conversion Rules

Before applying unit conversions to the full measurement dataset, representative values are used to verify that each conversion rule produces physically consistent results.

This separates transformation-rule validation from large-scale data processing.

In [96]:
conversion_tests = pd.DataFrame(
    [
        {
            "mnemonic": "DEPTH",
            "source_unit": "0.1 in",
            "source_value": 1000.0,
        },
        {
            "mnemonic": "TNPH",
            "source_unit": "PU",
            "source_value": 25.0,
        },
    ]
)

conversion_tests

,mnemonic,source_unit,source_value
0,DEPTH,0.1 in,1000.0
1,TNPH,PU,25.0


In [97]:
conversion_tests = conversion_tests.merge(
    conversion_rules,
    on=["mnemonic", "source_unit"],
    how="left",
)

In [98]:
conversion_tests["converted_value"] = (
    conversion_tests["source_value"]
    * conversion_tests["conversion_factor"]
    + conversion_tests["conversion_offset"]
)

conversion_tests[
    [
        "mnemonic",
        "source_value",
        "source_unit",
        "converted_value",
        "target_unit",
    ]
]

,mnemonic,source_value,source_unit,converted_value,target_unit
0,DEPTH,1000.0,0.1 in,2540.00,mm
1,TNPH,25.0,PU,0.25,v/v


In [99]:
depth_test = conversion_tests.loc[
    conversion_tests["mnemonic"] == "DEPTH",
    "converted_value"
].iloc[0]

tnph_test = conversion_tests.loc[
    conversion_tests["mnemonic"] == "TNPH",
    "converted_value"
].iloc[0]

assert np.isclose(depth_test, 2540.0)
assert np.isclose(tnph_test, 0.25)

print("DEPTH conversion validated.")
print("TNPH conversion validated.")
print("All conversion tests passed.")

DEPTH conversion validated.
TNPH conversion validated.
All conversion tests passed.
